In [1]:
import pandas as pd
import numpy as np
import requests
import time
import os
import random
import warnings

warnings.filterwarnings('ignore')

base_dir = r"C:\Users\Abhishek Shandilya\Desktop\suraksha"
API_URL = "http://localhost:8000"

print("Demo script ready.")
print("Backend URL:", API_URL)

Demo script ready.
Backend URL: http://localhost:8000


In [3]:
df = pd.read_csv(os.path.join(base_dir, "data", "processed", "transactions_scored.csv"))
df["timestamp"] = pd.to_datetime(df["timestamp"])

response = requests.get(f"{API_URL}/health")
print("Backend status:", response.json())
print("Transactions available:", len(df))
print("High risk transactions:", len(df[df["anomaly_score"] >= 0.8]))
print("Medium risk transactions:", len(df[(df["anomaly_score"] >= 0.5) & (df["anomaly_score"] < 0.8)]))

Backend status: {'status': 'healthy', 'timestamp': '2026-05-12T08:39:38.661751'}
Transactions available: 78299
High risk transactions: 267
Medium risk transactions: 2878


In [4]:
FEATURE_COLUMNS_API = [
    "txn_id", "user_id", "amount", "city", "lat", "lon", "device_id",
    "merchant_category", "km_from_last_txn", "minutes_from_last_txn",
    "impossible_speed", "hour", "is_weekend", "is_night", "txn_count_1hr",
    "txn_count_24hr", "amount_sum_1hr", "amount_sum_24hr", "amount_vs_user_avg",
    "is_new_device", "device_count_7d", "is_new_merchant_category"
]

def send_transaction(row):
    payload = {
        "txn_id": str(row["txn_id"]),
        "user_id": str(row["user_id"]),
        "amount": float(row["amount"]),
        "city": str(row["city"]),
        "lat": float(row["lat"]),
        "lon": float(row["lon"]),
        "device_id": str(row["device_id"]),
        "merchant_category": str(row["merchant_category"]),
        "km_from_last_txn": float(row["km_from_last_txn"]),
        "minutes_from_last_txn": float(row["minutes_from_last_txn"]),
        "impossible_speed": int(row["impossible_speed"]),
        "hour": int(row["hour"]),
        "is_weekend": int(row["is_weekend"]),
        "is_night": int(row["is_night"]),
        "txn_count_1hr": int(row["txn_count_1hr"]),
        "txn_count_24hr": int(row["txn_count_24hr"]),
        "amount_sum_1hr": float(row["amount_sum_1hr"]),
        "amount_sum_24hr": float(row["amount_sum_24hr"]),
        "amount_vs_user_avg": float(row["amount_vs_user_avg"]),
        "is_new_device": int(row["is_new_device"]),
        "device_count_7d": int(row["device_count_7d"]),
        "is_new_merchant_category": int(row["is_new_merchant_category"]),
    }
    response = requests.post(f"{API_URL}/transaction/score", json=payload)
    return response.json()

print("Transaction sender ready.")

Transaction sender ready.


In [7]:
normal_txns = df[df["anomaly_score"] < 0.5].head(50)
fraud_txns = df[df["anomaly_score"] >= 0.8].head(5)

print("Starting live transaction stream...")
print("Open http://localhost:3000 and watch the dashboard.")
print()
print("Phase 1 — Streaming normal transactions...")
print()

for idx, (_, row) in enumerate(normal_txns.iterrows()):
    result = send_transaction(row)
    status = result.get("action", "unknown")
    score = result.get("anomaly_score", 0)
    print(f"TXN {idx+1:02d} | {row['txn_id']} | {row['city']:12s} | INR {row['amount']:>10.0f} | Score: {score:.2f} | {status}")
    time.sleep(0.3)

print()
print("Phase 2 — Injecting fraud transactions...")
print()

for idx, (_, row) in enumerate(fraud_txns.iterrows()):
    result = send_transaction(row)
    status = result.get("action", "unknown")
    score = result.get("anomaly_score", 0)
    print(f"FRAUD {idx+1:02d} | {row['txn_id']} | {row['city']:12s} | INR {row['amount']:>10.0f} | Score: {score:.2f} | {status}")
    time.sleep(0.5)

print()
print("Demo simulation complete.")
print("Check the dashboard — flagged transactions are now visible.")

Starting live transaction stream...
Open http://localhost:3000 and watch the dashboard.

Phase 1 — Streaming normal transactions...

TXN 01 | TXN00000002 | Delhi        | INR       5881 | Score: 0.43 | approved
TXN 02 | TXN00000003 | Delhi        | INR       7838 | Score: 0.25 | approved
TXN 03 | TXN00000004 | Delhi        | INR      10018 | Score: 0.29 | approved
TXN 04 | TXN00000005 | Delhi        | INR       5642 | Score: 0.26 | approved
TXN 05 | TXN00000006 | Delhi        | INR       5643 | Score: 0.30 | approved
TXN 06 | TXN00000007 | Delhi        | INR      10158 | Score: 0.37 | approved
TXN 07 | TXN00000008 | Delhi        | INR       8137 | Score: 0.38 | approved
TXN 08 | TXN00000009 | Delhi        | INR       5056 | Score: 0.31 | approved
TXN 09 | TXN00000010 | Delhi        | INR       7577 | Score: 0.27 | approved
TXN 10 | TXN00000011 | Delhi        | INR       5072 | Score: 0.29 | approved
TXN 11 | TXN00000012 | Delhi        | INR       5066 | Score: 0.33 | approved
TXN 12 | 